# Exploratory Data Analysis with Pyspark and Spark SQL

The following notebook utilizes New York City taxi data from [TLC Trip Record Data](https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page)

## Instructions

- Load and explore nyc taxi data from january 0f 2019. The exercises can be executed using pyspark or spark sql (a subset of the questions will be re-answered using the language not chosen for the  main work).
- Load the zone lookup table to answer the questions about the nyc boroughs.  
- Load nyc taxi data from January of 2025 and compare data.  
- With any remaining time, work on the where to go from here section.
- Note: the initial lab is opened as read only. To save work completed utilize the `save notebook as` option and give the lab a new name.

In [85]:
import requests

# start a spark session and create a spark context
from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName("nyc_taxi") \
    .getOrCreate()

sc = spark.sparkContext

In [86]:
# set dl url for January 2019 trip data
download_url = 'https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2019-01.parquet'

# get the data
response = requests.get(download_url)

# check that response was good and save the data
jan_2019_trip_data = "yellow_tripdata_2019-01.parquet"
if response.status_code == 200:
    # Changed 'response' to 'data' here
    with open(jan_2019_trip_data, "wb") as f:
        f.write(response.content)


In [87]:
# create the dataframe
df_trips = spark.read.parquet(jan_2019_trip_data)

# A brief note on handling data sources in spark

The command above works well for loading data from parquet files because parquet is a self descibing file format, meaning that the metadata needed to build the dataframe is included directly in the format. However, when working with other formats such as csv or json, a schema must be provided or infered. In production code the schema should always be explicitly provided but during the data exploration phase it is acceptable to infer the schema, and when infering the schema it often best to use `.option("samplingRatio", <small-portion-of-data>)` to avoid using the entire dataset for schema inference.

```python
df_trips = spark.read.format("csv") \
    .option("header", "true") \
    .option("sep", ",") \
    .option("samplingRatio", 0.01) \
    .load("large_dataset.csv")
```

In [4]:
# Show the dataframe
df_trips.show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|       1| 2019-01-01 00:46:40|  2019-01-01 00:53:20|            1.0|          1.5|       1.0|                 N|         151|         239|           1|        7.0|  0.5|    0.5|      1.6

## Lab

### Part 1
This section can be completed either using pyspark commands or sql commands ( There will be a section after in which a self-chosen subset of the questions are re-answered using the language not used for the main section. i.e. if pyspark is chosen for the main lab, sql should be used to repeat some of the questions. )

- Add a column that creates a unique key to identify each record in order to answer questions about individual trips
- Which trip has the highest passanger count
- What is the Average passanger count
- Shortest/longest trip by distance? by time?.
- busiest day/slowest single day
- busiest/slowest time of day ( you may want to bucket these by hour or create timess such as morning, afternoon, evening, late night )
- On average which day of the week is slowest/busiest
- Does trip distance or num passangers affect tip amount
- What was the highest "extra" charge and which trip
- Are there any datapoints that seem to be strange/outliers (make sure to explain your reasoning in a markdown cell)?

In [25]:
#add a column with unique key
from pyspark.sql.functions import monotonically_increasing_id

df_trips = df_trips.withColumn(
    "trip_id",
    monotonically_increasing_id()
)
df_trips.show(5)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|    trip_id|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+-----------+
|       1| 2019-01-01 00:46:40|  2019-01-01 00:53:20|            1.0|          1.5|       1.0|                 N|         151|         239|           1

In [23]:
#highest passenger count
#

spark.sql("SELECT {df.passenger_count} FROM {df} order by {df.passenger_count} DESC", df=df_trips).show(1)

+---------------+
|passenger_count|
+---------------+
|            9.0|
+---------------+
only showing top 1 row


In [44]:
#average passenger count
spark.sql("SELECT AVG({df.passenger_count}) from {df}",df=df_trips).show()

+--------------------+
|avg(passenger_count)|
+--------------------+
|  1.5670317144945614|
+--------------------+



In [33]:
#Shortest/longest trip by distance? by time
spark.sql("Select  min({df.trip_distance}) as min, max({df.trip_distance}) as max from {df} ",df = df_trips).show()

+---+-----+
|min|  max|
+---+-----+
|0.0|831.8|
+---+-----+



In [43]:
#busiest day/slowest single day
spark.sql("select COUNT(*) as total, TO_DATE({df.tpep_pickup_datetime}) as date  from {df} group by TO_DATE({df.tpep_pickup_datetime}) order by total desc",df=df_trips).show(1)
spark.sql("select COUNT(*) as total, TO_DATE({df.tpep_pickup_datetime}) as date  from {df} group by TO_DATE({df.tpep_pickup_datetime}) order by total asc",df=df_trips).show(1)  



+------+----------+
| total|      date|
+------+----------+
|292499|2019-01-25|
+------+----------+
only showing top 1 row
+-----+----------+
|total|      date|
+-----+----------+
|    1|2019-05-20|
+-----+----------+
only showing top 1 row


In [48]:
#busiest/slowest time of day ( you may want to bucket these by hour or create timess such as morning, afternoon, evening, late night )
spark.sql("select HOUR({df.tpep_pickup_datetime}) as hour, count(*) AS total from {df} group by hour order by total DESC",df=df_trips).show()

+----+------+
|hour| total|
+----+------+
|  18|515390|
|  19|475186|
|  17|468479|
|  15|452691|
|  14|433139|
|  20|423156|
|  16|420843|
|  21|409901|
|  13|404153|
|  12|401173|
|  11|375441|
|   8|373742|
|  22|369041|
|   9|365935|
|  10|361390|
|   7|304858|
|  23|281941|
|   0|207842|
|   6|178598|
|   1|149254|
+----+------+
only showing top 20 rows


In [70]:
#On average which day of the week is slowest/busiest
        
spark.sql("""SELECT case when WEEKDAY({df.tpep_pickup_datetime}) = 0 then 'Monday'
when WEEKDAY({df.tpep_pickup_datetime}) = 2 then 'Tuesday'
when WEEKDAY({df.tpep_pickup_datetime}) = 3 then 'Wednesday'
when WEEKDAY({df.tpep_pickup_datetime}) = 4 then 'Thursday'
when WEEKDAY({df.tpep_pickup_datetime}) = 5 then 'Friday'
when WEEKDAY({df.tpep_pickup_datetime}) = 6 then 'Saturday'
else 'Sunday' end as day, count(*) as total from {df} group by day order by total desc""",df=df_trips).show()

+---------+-------+
|      day|  total|
+---------+-------+
|Wednesday|1357043|
|  Tuesday|1265264|
|   Sunday|1209084|
| Thursday|1087215|
|   Friday|1009985|
|   Monday| 908121|
| Saturday| 859905|
+---------+-------+



In [92]:
#Does trip distance or num passangers affect tip amount
spark.sql("""SELECT
    CASE
        WHEN {df.trip_distance} < 1 THEN '0-1'
        WHEN {df.trip_distance} < 3 THEN '1-3'
        WHEN {df.trip_distance} < 5 THEN '3-5'
        WHEN {df.trip_distance} < 10 THEN '5-10'
        ELSE '10+ miles'
    END AS distance_range,
    ROUND(AVG({df.tip_amount}), 2) AS avg_tip 
FROM {df}
GROUP BY
    distance_range ORDER BY avg_tip DESC""",df=df_trips).show()

+--------------+-------+
|distance_range|avg_tip|
+--------------+-------+
|     10+ miles|   6.22|
|          5-10|   3.47|
|           3-5|   2.27|
|           1-3|   1.45|
|           0-1|   0.94|
+--------------+-------+



In [98]:
#What was the highest "extra" charge and which trip
spark.sql("select {df.extra} as extra, {df.tip_amount} from {df} order by extra desc",df=df_trips).show(1)

+------+----------+
| extra|tip_amount|
+------+----------+
|535.38|       0.0|
+------+----------+
only showing top 1 row


In [100]:
#- Are there any datapoints that seem to be strange/outliers (make sure to explain your reasoning in a markdown cell)?
df_trips.select("VendorID","passenger_count","trip_distance","RatecodeID","store_and_fwd_flag","PULocationID","DOLocationID","payment_type","fare_amount").describe().show()

+-------+------------------+------------------+------------------+------------------+------------------+-----------------+------------------+-------------------+-----------------+
|summary|          VendorID|   passenger_count|     trip_distance|        RatecodeID|store_and_fwd_flag|     PULocationID|      DOLocationID|       payment_type|      fare_amount|
+-------+------------------+------------------+------------------+------------------+------------------+-----------------+------------------+-------------------+-----------------+
|  count|           7696617|           7667945|           7696617|           7667945|           7667945|          7696617|           7696617|            7696617|          7696617|
|   mean| 1.638174148460291|1.5670317144945614|2.8301461681153532|1.0583710498705976|              NULL|165.4004564602864|163.62890890894013|  1.286946979432652|12.52967677747685|
| stddev|0.5393984482313224|1.2244198591042095| 3.774548394256295|0.6780839328761981|              N

In [101]:
#- Are there any datapoints that seem to be strange/outliers (make sure to explain your reasoning in a markdown cell)?
df_trips.select("extra","mta_tax","tip_amount","tolls_amount","improvement_surcharge","total_amount","congestion_surcharge","airport_fee").describe().show()

+-------+------------------+------------------+------------------+------------------+---------------------+-----------------+--------------------+-----------+
|summary|             extra|           mta_tax|        tip_amount|      tolls_amount|improvement_surcharge|     total_amount|congestion_surcharge|airport_fee|
+-------+------------------+------------------+------------------+------------------+---------------------+-----------------+--------------------+-----------+
|  count|           7696617|           7696617|           7696617|           7696617|              7696617|          7696617|             2811730|          0|
|   mean|0.3374054146126797|0.4964963061043573|1.8208300763883147|0.3229543291538946|    0.299340957215541|15.81065134371489|3.289789560164027...|       NULL|
| stddev|0.5313564053935059|0.0549238203069398|2.4994631914320986|2.0298119241900534| 0.019077043464518655|261.8117056584905|0.009068830463889703|       NULL|
|    min|             -60.0|              -0.5

outliers data will be rows with :
- passenger count = 0 ; meaning no client in the cab
- trip distance = 0; canceled ride maybe or gps bug. can be used but can cause problem when multipying or dividing by zero. Also any distance over 60 (miles) is suspicious
- rate code mean is close to 1 yet we have values up to 99. since I dont know exaclty what it is refering to it could be rare or an error
- fare amount below 0  : cost being negative can be tretaed as reimboursement, or a bug
- extra being negative is also not possible
- tip amount begin negative could also be a reimboursement or outlier
- same for toll amount,total amount (also 3000+ on just tolls can be suspected as  an outlier)
- no airport fee is  also weird, coudl be a bug or based on rides not near the airport

### Part 2

- Using the code for loading the first dataset as an example, load in the taxi zone lookup and answer the following questions
- which borough had most pickups? dropoffs?
- what are the busy/slow times by borough 
- what are the busiest days of the week by borough?
- what is the average trip distance by borough?
- what is the average trip fare by borough?
- highest/lowest faire amounts for a trip, what burough is associated with the each
- load the dataset from the most recently available january, is there a change to any of the average metrics.

In [114]:
zone_url = 'https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv'
zone_response = requests.get(zone_url)

zone_file = "taxi_zone_lookup.csv"
if zone_response.status_code == 200:
    with open(zone_file, "wb") as f:
        f.write(zone_response.content)

df_zones = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(zone_file)

#join to group the datasets
df_trips_joined = df_trips.join(
    df_zones, 
    df_trips.PULocationID == df_zones.LocationID, 
    "left"
).withColumnRenamed("Borough", "pickup_borough") \
 .withColumnRenamed("Zone", "pickup_zone") \
 .drop("LocationID", "service_zone")


df_trips_joined.select("pickup_borough", "pickup_zone", "trip_distance").show(5)

+--------------+--------------------+-------------+
|pickup_borough|         pickup_zone|trip_distance|
+--------------+--------------------+-------------+
|     Manhattan|    Manhattan Valley|          1.5|
|     Manhattan|Upper West Side S...|          2.6|
|     Manhattan|Upper East Side N...|          0.0|
|        Queens|Queensbridge/Rave...|          0.0|
|        Queens|Queensbridge/Rave...|          0.0|
+--------------+--------------------+-------------+
only showing top 5 rows


In [112]:
from pyspark.sql.functions import count, avg, desc

# 1. Joindre aussi pour le dropoff borough pour pouvoir répondre aux deux
df_trips_fully_joined = df_trips_joined.join(
    df_zones, 
    df_trips.DOLocationID == df_zones.LocationID, 
    "left"
).withColumnRenamed("Borough", "dropoff_borough") \
 .withColumnRenamed("Zone", "dropoff_zone") \
 .drop("LocationID", "service_zone")

# 2. Borough avec le plus de Pickups
print("--- Pickups par Borough ---")
df_trips_fully_joined.groupBy("pickup_borough").agg(count("*").alias("pickup_count")) \
                     .orderBy(desc("pickup_count")).show()

# 3. Borough avec le plus de Dropoffs
print("--- Dropoffs par Borough ---")
df_trips_fully_joined.groupBy("dropoff_borough").agg(count("*").alias("dropoff_count")) \
                     .orderBy(desc("dropoff_count")).show()

# 4. Distance moyenne et tarif moyen par Pickup Borough
print("--- Distance et Tarif moyens par Pickup Borough ---")
df_trips_fully_joined.groupBy("pickup_borough") \
                     .agg(avg("trip_distance").alias("avg_distance"), 
                          avg("total_amount").alias("avg_fare")) \
                     .orderBy(desc("avg_distance")).show()

--- Pickups par Borough ---
+--------------+------------+
|pickup_borough|pickup_count|
+--------------+------------+
|     Manhattan|     6950965|
|        Queens|      471173|
|       Unknown|      159815|
|      Brooklyn|       91905|
|         Bronx|       18062|
|           N/A|        3890|
|           EWR|         446|
| Staten Island|         361|
+--------------+------------+

--- Dropoffs par Borough ---
+---------------+-------------+
|dropoff_borough|dropoff_count|
+---------------+-------------+
|      Manhattan|      6817355|
|         Queens|       340972|
|       Brooklyn|       301105|
|        Unknown|       149097|
|          Bronx|        58085|
|            N/A|        16904|
|            EWR|        10914|
|  Staten Island|         2185|
+---------------+-------------+

--- Distance et Tarif moyens par Pickup Borough ---
+--------------+------------------+------------------+
|pickup_borough|      avg_distance|          avg_fare|
+--------------+------------------+

In [115]:
url_2025 = 'https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-01.parquet'
resp_2025 = requests.get(url_2025)

file_2025 = "yellow_tripdata_2025-01.parquet"
if resp_2025.status_code == 200:
    with open(file_2025, "wb") as f:
        f.write(resp_2025.content)

df_2025 = spark.read.parquet(file_2025)


#Jan 2019
df_trips.select(avg("trip_distance").alias("avg_dist_2019"), avg("total_amount").alias("avg_fare_2019")).show()

+------------------+-----------------+
|     avg_dist_2019|    avg_fare_2019|
+------------------+-----------------+
|2.8301461681153532|15.81065134371489|
+------------------+-----------------+



In [116]:
#Jan 2025
df_2025.select(avg("trip_distance").alias("avg_dist_2025"), avg("total_amount").alias("avg_fare_2025")).show()

+-----------------+------------------+
|    avg_dist_2025|     avg_fare_2025|
+-----------------+------------------+
|5.855126178843539|25.611291697280986|
+-----------------+------------------+



### Part 3

- choose 3 questions from above and re-answer them using the language you did not use for the main notebook . (i.e - if you completed the exercise in python, redo 3 questions in pure sql) . at least one of the questions to be redone must involve a join

In [103]:
df_trips.select("passenger_count").sort(df_trips.passenger_count.desc()).show(1)

+---------------+
|passenger_count|
+---------------+
|            9.0|
+---------------+
only showing top 1 row


In [105]:
from pyspark.sql.functions import avg

df_trips.select(avg("passenger_count").alias("avg_passenger_count")).show()

+-------------------+
|avg_passenger_count|
+-------------------+
| 1.5670317144945614|
+-------------------+



In [109]:
from pyspark.sql.functions import col

df_trips.select("trip_distance", "tpep_pickup_datetime", "tpep_dropoff_datetime").orderBy(col("trip_distance").asc()).show(1)

df_trips.select("trip_distance", "tpep_pickup_datetime", "tpep_dropoff_datetime").orderBy(col("trip_distance").desc()).show(1)


+-------------+--------------------+---------------------+
|trip_distance|tpep_pickup_datetime|tpep_dropoff_datetime|
+-------------+--------------------+---------------------+
|          0.0| 2018-12-21 13:48:30|  2018-12-21 13:52:40|
+-------------+--------------------+---------------------+
only showing top 1 row
+-------------+--------------------+---------------------+
|trip_distance|tpep_pickup_datetime|tpep_dropoff_datetime|
+-------------+--------------------+---------------------+
|        831.8| 2019-01-25 21:56:39|  2019-01-25 22:06:08|
+-------------+--------------------+---------------------+
only showing top 1 row


# Where to go from here

- Continue building the dataset by loading in more data, start by completing the data for 2019 and calculating the busiest season (fall, winter, spring, summer)
- As of spark v4 dataframes have native visualization support. Choose at least 3 questions from above and provide visualizations.
- Explore a dataset/datasets of your choosing